# Predictive Maintenance Classification Pipeline

This notebook is a guided implementation template for the Module 1 assessment.  
It follows the required execution order: EDA, data preparation, feature engineering, train/test split, training-set-only resampling, KNN scaling, Decision Tree without scaling, hyperparameter comparison, overfitting analysis, and final accuracy-based verdict.

Renan de Brito Leme

## 0. Environment Setup

In [ ]:
# ============================================================
# RF00 - Libraries import
# ============================================================

import numpy as np
import pandas as pd
import os

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Resampling
from imblearn.over_sampling import SMOTE
# Alternative if SMOTE is too slow or unavailable:
# from imblearn.under_sampling import RandomUnderSampler

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
sns.set_context("notebook")

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Environment configured successfully.")

## 1. Load the Dataset

In [ ]:
DATA_PATH = "predictive_maintenance.csv"
raw_dataset = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully.")
print("Shape:", raw_dataset.shape)
raw_dataset.head()

## 2. Data Dictionary and Modeling Assumptions

Important modeling assumptions:

1. `machine_failure` is the binary target variable.
2. The columns `tool_wear_failure`, `heat_dissipation_failure`, `power_failure`, `overstrain_failure`, and `random_failure` are specific failure causes and must not be used as predictors because they would leak information about the target.
3. `udi` and `product_id` are identifiers, not physical process variables, so they should also be excluded from the model.
4. The relevant predictors are equipment type and sensor/process variables.

In [ ]:
target_column = "machine_failure"

identifier_columns = ["unique_id", "product_id"]

leakage_columns = [
    "tool_wear_failure",
    "heat_dissipation_failure",
    "power_failure",
    "overstrain_failure",
    "random_failure",
]

numeric_sensor_features = [
    "air_temperature_k",
    "process_temperature_k",
    "rotational_speed_rpm",
    "torque_nm",
    "tool_wear_min",
]

categorical_features = ["machine_type"]

print("Target column:", target_column)
print("Identifier columns:", identifier_columns)
print("Leakage columns:", leakage_columns)
print("Numeric sensor features:", numeric_sensor_features)
print("Categorical features:", categorical_features)

# Phase 1 — Exploratory Data Analysis (EDA)

Required outputs:

- Dataset dimensions.
- Variable data types.
- Descriptive statistics with `.describe()`.
- At least three analytical plots.
- A written interpretation connecting the EDA findings to modeling decisions.

In [ ]:
print("Dataset shape:", raw_dataset.shape)
raw_dataset.info()
print("-------------------------------------------------------")
print("Data types:")
print(raw_dataset.dtypes)

In [ ]:
raw_dataset.describe().T

In [ ]:
# Missing values overview
missing_summary = pd.DataFrame({
    "missing_count": raw_dataset.isna().sum(),
    "missing_percent": raw_dataset.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

missing_summary

In [ ]:
# Target distribution
target_distribution = raw_dataset[target_column].value_counts().sort_index()

target_percentage = (
    raw_dataset[target_column]
    .value_counts(normalize=True)
    .sort_index() * 100
)

target_summary = pd.DataFrame({
    "count": target_distribution,
    "percentage": target_percentage.round(2),
})

target_summary

In [ ]:
# Plot 1: target imbalance
plt.figure(figsize=(6, 4))
sns.countplot(data=raw_dataset, x=target_column)
plt.title("Target Variable Distribution")
plt.xlabel("Machine Failure: 0 = Normal, 1 = Failure")
plt.ylabel("Count")
output_dir="outputs"
file_path = os.path.join(output_dir, "target_variable_distribution.png")
plt.savefig(file_path, dpi=150)
plt.show()
plt.close()
print(f"  Chart exported: {file_path}")

In [ ]:
# Plot 2: numerical variable distributions
raw_dataset[numeric_sensor_features].hist(figsize=(12, 8), bins=30)
plt.suptitle("Distribution of Numerical Predictive Variables")
plt.tight_layout()
output_dir="outputs"
file_path = os.path.join(output_dir, "distribution_of_numerical_predictive_variables.png")
plt.savefig(file_path, dpi=150)
plt.show()
plt.close()
print(f"  Chart exported: {file_path}")

In [ ]:
# Plot 3: Pearson correlation heatmap
correlation_columns = numeric_sensor_features + [target_column]
correlation_matrix = raw_dataset[correlation_columns].corr(numeric_only=True)

plt.figure(figsize=(9, 6))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Pearson Correlation Heatmap")
plt.tight_layout()
output_dir="outputs"
file_path = os.path.join(output_dir, "pearson_correlation_heatmap.png")
plt.savefig(file_path, dpi=150)
plt.show()
plt.close()
print(f"  Chart exported: {file_path}")

In [ ]:
# Plot 4: equipment type vs target
plt.figure(figsize=(7, 4))
sns.countplot(
    data=raw_dataset,
    x="machine_type",
    hue=target_column
)
plt.title("Machine Type vs Machine Failure")
plt.xlabel("Machine Type")
plt.ylabel("Count")
plt.legend(title="Failure")
plt.tight_layout()
output_dir="outputs"
file_path = os.path.join(output_dir, "machine_type_vs_machine_failure.png")
plt.savefig(file_path, dpi=150)
plt.show()
plt.close()
print(f"  Chart exported: {file_path}")

### EDA Interpretation

- The target variable is highly imbalanced, so resampling will be required after the train/test split.
- Missing values appear in sensor variables, so imputation is required before modeling.
- Because some numerical features may be skewed and industrial sensor data can contain outliers, median imputation is a robust option.
- KNN will require scaling because it depends on distance calculations.
- Decision Trees do not require scaling because they split variables by thresholds.
- Failure-cause columns must be excluded to prevent data leakage.

# Phase 2 — Data Cleaning and Preparation

Required outputs:

- Identify and remove duplicated rows.
- Identify missing values.
- Apply mean or median imputation with a written justification.
- Create boxplots to inspect outliers in explanatory variables.

In [ ]:
# Duplicate rows
duplicate_count = raw_dataset.duplicated().sum()
print("Duplicate rows before cleaning:", duplicate_count)

clean_dataset = raw_dataset.drop_duplicates().copy()
print("Shape before duplicate removal:", raw_dataset.shape)
print("Shape after duplicate removal:", clean_dataset.shape)

In [ ]:
# Missing values after duplicate removal
clean_dataset.isna().sum().sort_values(ascending=False)

In [ ]:
# Boxplots for outlier inspection before imputation
for feature in numeric_sensor_features:
    plt.figure(figsize=(7, 4))
    sns.boxplot(
    data=clean_dataset,
    x=feature
    )
    plt.title(f"Boxplot - {feature}")
    plt.xlabel(feature)
    plt.tight_layout()
    output_dir="outputs"
    file_path = os.path.join(output_dir, f"Boxplot_{feature}.png")
    plt.savefig(file_path, dpi=150)
    plt.show()
    plt.close()
    print(f"  Chart exported: {file_path}")

### Outlier Decision

Outliers were retained because unusual sensor values may represent meaningful abnormal operating conditions associated with failure rather than data-entry errors.

# Phase 3 — Train/Test Split Before Learned Transformations

Required outputs:

- Separate predictors `X` from target `y`.
- Split into training and test sets using 80/20.
- Use `stratify=y`.
- Apply resampling only to the training data to avoid data leakage.

In [ ]:
base_feature_columns=numeric_sensor_features+categorical_features

X_raw=clean_dataset[base_feature_columns].copy()
y=clean_dataset[target_column].copy()

print("Base predictor columns:",base_feature_columns)
print("X_raw shape:",X_raw.shape);
print("y shape:",y.shape)

In [ ]:
X_train_raw,X_test_raw,y_train,y_test=train_test_split(
    X_raw,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("X_train_raw shape:",X_train_raw.shape)
print("X_test_raw shape:",X_test_raw.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

# Phase 4 — Training-Fitted Imputation

In [ ]:
# Phase 4 — Training-Fitted Imputationnumeric_imputer=SimpleImputer(strategy="median")
numeric_imputer=SimpleImputer(strategy="median")
categorical_imputer=SimpleImputer(strategy="most_frequent")

X_train_imputed=X_train_raw.copy();
X_test_imputed=X_test_raw.copy();

X_train_imputed[numeric_sensor_features]=numeric_imputer.fit_transform(X_train_raw[numeric_sensor_features])
X_test_imputed[numeric_sensor_features]=numeric_imputer.transform(X_test_raw[numeric_sensor_features])

X_train_imputed[categorical_features]=categorical_imputer.fit_transform(X_train_raw[categorical_features])
X_test_imputed[categorical_features]=categorical_imputer.transform(X_test_raw[categorical_features])

print("Missing after imputation - train:",int(X_train_imputed.isna().sum().sum()))
print("Missing after imputation - test:",int(X_test_imputed.isna().sum().sum()))

In [ ]:
imputation_summary=pd.DataFrame({
    "feature":numeric_sensor_features,
    "training_median":numeric_imputer.statistics_
})
imputation_summary

### Imputation Justification

The imputers are fitted only on training data. Median imputation is used for numerical variables because it is less affected by extreme values than the mean. The most frequent category is used for `machine_type`.

# Phase 5 — Feature Engineering After Imputation

The new feature `power_proxy` is created using the suggested mathematical operation:

`power_proxy = rotational_speed_rpm * torque_nm`# Phase 3 - Feature Engineering


An additional feature, `temperature_difference_k`, is also created to represent the difference between process temperature and air temperature.

In [ ]:
def add_engineered_features(dataset: pd.DataFrame) -> pd.DataFrame:
    result=dataset.copy()
    result["power_proxy"]=result["rotational_speed_rpm"]*result["torque_nm"]
    result["temperature_difference_k"]=result["process_temperature_k"]-result["air_temperature_k"]
    return result

X_train_engineered=add_engineered_features(X_train_imputed)
X_test_engineered=add_engineered_features(X_test_imputed)
engineered_features=["power_proxy","temperature_difference_k"]
print("Engineered features:",engineered_features)

In [ ]:
for name,dataset in {"training":X_train_engineered,"test":X_test_engineered}.items():
    missing=int(dataset[engineered_features].isna().sum().sum())
    infinite=int(np.isinf(dataset[engineered_features]).sum().sum())
    print(f"{name.capitalize()} - missing: {missing}, infinite: {infinite}")
X_train_engineered[engineered_features].describe().T

### Feature Engineering Interpretation

The `power` feature combines rotational speed and torque. In an industrial maintenance context, the combination of high rotational speed and high torque can indicate higher mechanical demand. The `temperature_difference` feature measures the difference between process temperature and ambient air temperature, which can help represent heat generation during operation.

# Phase 6 — Training-Fitted One-Hot Encoding

The categorical feature `machine_type` has three categories: `L`, `M`, and `H`.

Using One-Hot Encoding to create only two columns:

- `machine_type_L`
- `machine_type_M`

or another pair depending on the alphabetical order of the categories in the dataset.

The removed category becomes the reference category. If both dummy columns are zero, the machine belongs to the reference type.

In [ ]:
one_hot_encoder=OneHotEncoder(
    categories=[["L","M","H"]],drop=["L"],handle_unknown="ignore",sparse_output=False,dtype=int
)
train_encoded_array=one_hot_encoder.fit_transform(X_train_engineered[categorical_features])
test_encoded_array=one_hot_encoder.transform(X_test_engineered[categorical_features])
one_hot_features=one_hot_encoder.get_feature_names_out(categorical_features).tolist()
X_train_encoded=pd.DataFrame(train_encoded_array,columns=one_hot_features,index=X_train_engineered.index)
X_test_encoded=pd.DataFrame(test_encoded_array,columns=one_hot_features,index=X_test_engineered.index)
print("One-Hot columns:",one_hot_features)

In [ ]:
continuous_features=numeric_sensor_features+engineered_features
X_train_prepared=pd.concat([X_train_engineered[continuous_features],X_train_encoded],axis=1)
X_test_prepared=pd.concat([X_test_engineered[continuous_features],X_test_encoded],axis=1)
print("Prepared training shape:",X_train_prepared.shape)
print("Prepared test shape:",X_test_prepared.shape)
print(X_train_prepared.columns.tolist())

### Training-Fitted One-Hot Encoding Interpretation

The categorical variable machine_type was transformed using One-Hot Encoding. This technique converts the categorical machine types into binary variables (machine_type_M and machine_type_H), allowing machine learning algorithms to process categorical information without imposing any ordinal relationship between categories. The L category was explicitly selected as the reference category (drop=["L"]), so observations with both dummy variables equal to zero correspond to machines of type L. This approach avoids redundant information (dummy variable trap), reduces multicollinearity, and enables the models to evaluate the influence of each machine type on the probability of machine failure.

# Phase 7 — Decision Tree Branch and KNN Branch

Required outputs:

- Apply `StandardScaler` only for KNN.
- Use `fit_transform` on training data and `transform` on test data.
- Keep Decision Tree data without scaling.
- Explain why Decision Trees are scale-insensitive.

In [ ]:
# Decision Tree Branch
smote_tree=SMOTE(random_state=RANDOM_STATE)

X_train_tree,y_train_tree=smote_tree.fit_resample(X_train_prepared,y_train)
X_test_tree=X_test_prepared.copy()

print("Before SMOTE:")
print(y_train.value_counts())
print("Tree training shape before SMOTE:", X_train_prepared.shape)

print("\nAfter SMOTE:")
print(y_train_tree.value_counts())
print("Tree training shape after SMOTE:", X_train_tree.shape)

print("Tree test shape:", X_test_tree.shape)

In [ ]:
# KNN Branch
scaler=StandardScaler()

X_train_knn_scaled=X_train_prepared.copy();
X_test_knn=X_test_prepared.copy()

X_train_knn_scaled[continuous_features]=scaler.fit_transform(
    X_train_prepared[continuous_features])
X_test_knn[continuous_features]=scaler.transform(
    X_test_prepared[continuous_features])

smote_knn=SMOTE(random_state=RANDOM_STATE)
X_train_knn,y_train_knn=smote_knn.fit_resample(
    X_train_knn_scaled,
    y_train)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_knn.value_counts())

X_train_knn_scaled[continuous_features].agg(["mean", "std"]).T

### Scaling and Resampling Justification

The KNN scaler is fitted on real training observations before SMOTE, so synthetic samples do not influence the learned mean and standard deviation. SMOTE is then applied in the scaled feature space. The Decision Tree uses encoded but unscaled data because threshold-based splits are insensitive to feature scale.

# Phase 8 — Hyperparameter Tuning and Overfitting Analysis

Required outputs:

- Train KNN using at least three odd values of `n_neighbors`.
- Train Decision Tree using at least three values of `max_depth`.
- Record training and test accuracy.
- Identify overfitting points and the most stable configuration.

In [ ]:
knn_results=[]

for neighbors in [3,5,7,9,11]:
    model=KNeighborsClassifier(
        n_neighbors=neighbors
    )
    model.fit(X_train_knn,y_train_knn)
    train_pred=model.predict(X_train_knn);
    test_pred=model.predict(X_test_knn)
    train_acc=accuracy_score(y_train_knn,train_pred);
    test_acc=accuracy_score(y_test,test_pred)
    knn_results.append({
        "model":"KNN",
        "n_neighbors":neighbors,
        "train_accuracy":train_acc,
        "test_accuracy":test_acc,
        "accuracy_gap":train_acc-test_acc
    })
    
knn_results_df=pd.DataFrame(knn_results_df.round(4))
knn_results_df.sort_values(
    by="test_accuracy",
    ascending=False
)

In [ ]:
tree_results=[]

for depth in [3,5,7,10,None]:
    model=DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE
    )
    model.fit(X_train_tree,y_train_tree)
    train_pred=model.predict(X_train_tree);
    test_pred=model.predict(X_test_tree)
    train_acc=accuracy_score(y_train_tree,train_pred);
    test_acc=accuracy_score(y_test,test_pred)
    tree_results.append({
        "model":"Decision Tree",
        "max_depth":depth,
        "train_accuracy":train_acc,
        "test_accuracy":test_acc,
        "accuracy_gap":train_acc-test_acc})
    
tree_results_df=pd.DataFrame(tree_results).round(4)
tree_results_df.sort_values(
    by="test_accuracy",
    ascending=False
)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(knn_results_df["n_neighbors"],knn_results_df["train_accuracy"],marker="o",label="Train Accuracy")
plt.plot(knn_results_df["n_neighbors"],knn_results_df["test_accuracy"],marker="o",label="Test Accuracy")
plt.title("KNN - Train vs Test Accuracy"); plt.xlabel("Number of Neighbors"); plt.ylabel("Accuracy")
plt.legend();
plt.grid(True);
plt.tight_layout()
file_path=os.path.join(OUTPUT_DIR,"knn_train_vs_test_accuracy.png")
plt.savefig(file_path,dpi=150,bbox_inches="tight");
plt.show();
plt.close();
print(f"Chart exported: {file_path}")

In [ ]:
best_knn_row=knn_results_df.sort_values(["test_accuracy","accuracy_gap"],ascending=[False,True]).iloc[0]
best_tree_row=tree_results_df.sort_values(["test_accuracy","accuracy_gap"],ascending=[False,True]).iloc[0]
print("Best KNN configuration:");
print(best_knn_row)
print("\nBest Decision Tree configuration:");
print(best_tree_row)

### Overfitting Interpretation

The training and test accuracies were compared to assess the models' ability to generalize. A large difference between these values indicates overfitting, meaning the model has learned the training data too specifically and performs worse on unseen data. For KNN, very small values of n_neighbors increase model complexity and sensitivity to noise, while for Decision Trees, allowing unlimited depth (max_depth=None) can produce overly complex trees that memorize the training data. The most appropriate configuration is therefore the one that achieves high test accuracy while maintaining a small train-test accuracy gap, indicating a good balance between predictive performance and generalization.